# Epistemic Driven Epistemic Landscapes (EDEL) Pipeline

This notebook demonstrates the full modular EDEL pipeline, from data collection to interactive 3D landscapes.

## 0. Parameters and setup

In [6]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# EDEL Pipeline Imports
from edel.config.defaults import RUN_CONFIG
from edel.io.llm import get_llm_client
from edel.pipeline import (
    run_data_stage, run_structuring_stage, run_embedding_stage,
    run_projection_stage, run_vector_field_stage, run_clustering_stage,
    run_labeling_stage, run_landscape_stage
)
from edel.viz import (
    plot_abstract_length_dist, plot_publication_year_dist, plot_citation_dist,
    plot_projection_2d, plot_transition_signatures, plot_movement_magnitudes,
    plot_vector_field, plot_field_magnitude, plot_field_density,
    plot_clusters_on_landscape, plot_field_clusters, plot_cluster_trajectories,
    print_cluster_summaries, plot_epistemic_map,
    plot_landscape_3d, plot_landscape_contour
)

config = RUN_CONFIG.copy()
print("✅ Setup complete.")

ImportError: cannot import name 'run_data_stage' from 'edel.pipeline' (/home/lcorreia/edel/edel/pipeline/__init__.py)

In [ ]:
config = {

    "processing_mode": "batch", # "simple" | "batch"
    "embedding_mode": "aspects", # "aspects" | "documents"

    "data": {
        "provider": {
            "type": "openalex",
            "topic_id": "T10102",
            "topic_name": "Scientometrics",
            "region": None,
            "params": {
                "n_documents": 10,
                "avg_length": 150,
            }
         },
        "transforms": [
            {"type": "shuffle_words"}
        ]
    },

    "structured_abstracts": {
        "provider": "openai",
        "model": "gpt-5-mini",
        "min_sentences": 4,
        "min_tokens": 80,
    },

    "embedding": {
        "mode": "multi", # multi | single | abstract
        "provider": "openai",
        "model": "text-embedding-ada-002",
        "n_dimensions": 1536,
        "batch_size": 5000
    },

    "dimensionality_reduction": {
        "method": "diffusion",
        "n_neighbors": 15,
        "random_state": 0,
        "min_dist": 0.1,
        "metric": "cosine",
    },

    "vector_field": {
        "method": "diffusion",
        "grid_size": 25,
        "min_count": 3,
        "smooth_sigma": 1.0,
        "compute_divergence": True,
        "compute_magnitude": True,
    },

    "clustering": {
        "domain": {
            "source": "proj_p",
            "algorithm": "hdbscan",
            "params": {
                "min_cluster_size": 15,
            },
        },
        "field": {
            "source": "field",
            "algorithm": "hdbscan",
            "params": {
                "min_cluster_size": 10,
            },
        },
        "style": {
            "source": "features",
            "algorithm": "gmm",
            "params": {
                "n_components": 4,
            },
        },
        "operator": {
            "source": "operators",
            "algorithm": "hdbscan",
            "params": {
                "min_cluster_size": 20,
            },
        },
    },

    "labeling": {
        "provider": "openai",
        "model": "gpt-5-mini",
        "text_column": "abstract_text",
        "topic": None,
        "language": "en",
        "axis": {
            "enabled": True,
            "projection": "diffusion",
            "n_samples": 5,
        },
        "clusters": {
            "enabled": True,
            "cluster_keys": [
                "domain",
                "style",
                "operator",
                "field"
            ],
            "n_samples": 5,
        }
    },

    "landscape": {
      "metric": "cited_by_count",
      "log_scale": True,
      "grid": {
        "num_bins": 50,
        "sigma": 1.5
      },
      "scale": 12.0,
      "color_cluster": "cluster_domain",
      "style_cluster": "cluster_style",
      "field": {
        "enabled": True,
        "type": "total",
        "step": 2,
        "scale": 0.07,
        "width": 1
      }
    }
}

## 1. Data Collection

In [ ]:
print("Running Stage 1: Data Collection...")
df = run_data_stage(config)
print(f"Loaded {len(df)} documents.")

plot_abstract_length_dist(df)
plot_publication_year_dist(df)
plot_citation_dist(df)

## 2. Structuring

In [ ]:
print("Running Stage 2: Structuring abstracts...")
df = run_structuring_stage(df, config)
df.head(2)

## 3. Text Embeddings

In [ ]:
print("Running Stage 3: Embedding aspects...")
df = run_embedding_stage(df, config)
print("Embeddings computed.")

## 4. Dimensionality Reduction

In [ ]:
print("Running Stage 4: Projection...")
df = run_projection_stage(df, config)

method = config["dimensionality_reduction"]["method"]
plot_projection_2d(df, method=method, draw_arrows=True, arrow_step=20)
plot_transition_signatures(df)
plot_movement_magnitudes(df)

## 5. Vector Field

In [ ]:
print("Running Stage 5: Vector Field...")
field = run_vector_field_stage(df, config)

plot_vector_field(field, field_type="total", color_by_mag=True)
plot_field_magnitude(field, operator="pm")
plot_field_density(field)

## 6. Clustering

In [ ]:
print("Running Stage 6: Clustering...")
df, field = run_clustering_stage(df, field, config)

plot_clusters_on_landscape(df, cluster_key="domain")
plot_field_clusters(field, cluster_key="field")
plot_cluster_trajectories(df, cluster_key="style")

## 7. Labeling

In [ ]:
print("Running Stage 7: Labeling...")
llm_client = get_llm_client(config.get("labeling", {}))
label_results = run_labeling_stage(df, field, config, llm_client)

print_cluster_summaries(label_results, cluster_key="domain")

## 8. Landscape

In [ ]:
print("Running Stage 8: Landscape preparation...")
landscape_results = run_landscape_stage(df, field, config)

topic_name = config["data"]["provider"]["topic_name"]

# Final Epistemic Map
plot_epistemic_map(df, label_results, topic_name=topic_name)

# Interactive 2D Contour Map
plot_landscape_contour(df, landscape_results, field=field, topic_name=topic_name)

# Interactive 3D Surface
plot_landscape_3d(df, landscape_results, topic_name=topic_name)